In [102]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from ITIS_Model_funcs import get_nominal_param,OLS_res,ITIS

In [103]:
param_log,IC = get_nominal_param()
IC = [2.0, 0., 0.1595967, 0., 14.68266827,
          41.49891056, 39.97751646, 11.29827208]

D = '3'         # '1' for ACTH + Cort,
                # '2' for ACTH + Cort + TNF-a
                # '3' for ACTH + Cort + TNF-a + IL10
dpoints = '1'   # '1' for 25, '2' for 13
pickles = ['11', '12', '21', '22', '31', '32']

with open('OLS_Results\\ResAnalysis' + D + dpoints + '.pkl', 'rb') as f:
    results = pickle.load(f)

        ##Reference
        # all_results = {
        #     'output_ids': output_ids,
        #     'param_ids' : param_ids,
        #     'param_in' : param_in,
        #     'twonorms' : twonorms,
        #     't_data' : t_data,
        #     'y_data' : y_data,
        #     'opt_model' : opt_model,
        #     'true_sol' : true_sol,
        #     'param_opt' : param_opt,
        #     'optimal_solution' :np.exp(least_sq_sol.x),
        #     'objvalue' : least_sq_sol.cost
        # }


output_ids = results['output_ids']
t_data = results['t_data']
param_opt = results['param_opt']
y_data = results['y_data']
param_ids = results['param_ids']

##SENSITIVITY ANALYSIS
h = 1e-6  #amount to perturb parameters
n_param = len(param_opt)
n_states = len(output_ids)

S = np.zeros((n_param, len(t_data) * n_states)) ##Initialize shape of sensitivity matrix.

for i in range(n_param):  #calculate the relative residual sensitivity to each 45 parameters

        param_in = param_log[i]
        param_delta = param_in + h

        S[i, :] = ((1 / h) * (OLS_res(param_delta, y_data, t_data, i, output_ids, param_log, IC)
                                            - OLS_res(param_in, y_data, t_data, i, output_ids, param_log, IC)))

In [104]:
##import rankings from global analysis
with open('paramRankings\\rankingDesign' + D + '.pkl', 'rb') as f:
    results = pickle.load(f)

rank_value = results['rank_value']      # sorted ranking values
param_sorted = results['param_sorted']  #accordingly sorted params as strings
rank_cut = np.array(rank_value)[np.array(rank_value) > .25 * np.array(rank_value)[0]]
param_sorted = np.array(param_sorted)[:len(rank_cut)]

param_titles = ['d1',
'k1','k2','h1','h2','h3','d2',
'k3','k4','h4','d3',
'h5','h6','k5','k6','h7','d4',
'b1','k7','h8','k8','h9','d5','h10',
'b2','k9','k10','k11','d6',
'k12','k13','k14','h11','d7',
'k15','k16','d8',
'alpha', 'k', 'beta', 'L', 'eps', 'delta', 'T', 'Nc']

circadian_param = ['alpha', 'k', 'beta', 'L', 'eps', 'delta', 'T', 'Nc'] #exclude?

cond = 0
unid = []
p_indices = [param_titles.index(param_sorted[0])]
for i in range(1,len(rank_cut)):
    p_indices.append(param_titles.index(param_sorted[i]))
    S_opt = S[p_indices,:]
    F_opt = S_opt@S_opt.T
    cond = np.linalg.cond(F_opt)
    if cond > 1e+5:
        unid.append(p_indices.pop())
        print("Removed: ", unid[-1])


S_opt = S[p_indices,:]
F_opt = S_opt@S_opt.T
C_opt = np.linalg.inv(F_opt)
selected = [param_titles[i] for i in p_indices]
print("Design : ", D + dpoints)
print("Number of selected parameters:", len(p_indices))
print("Selected Parameters: ",  selected)
print(p_indices)
print("Excluded Parameters: ", [param_titles[i] for i in unid])
print("Condition Number of F:", np.linalg.cond(F_opt))
print("Diagonals of C:", np.diag(C_opt))

Design :  31
Number of selected parameters: 17
Selected Parameters:  ['h6', 'd7', 'k3', 'd4', 'd8', 'T', 'beta', 'h7', 'h11', 'k14', 'd5', 'k4', 'h4', 'k6', 'h9', 'd3', 'k1']
[12, 33, 7, 16, 36, 43, 39, 15, 32, 31, 22, 8, 9, 14, 21, 10, 1]
Excluded Parameters:  []
Condition Number of F: 26392.526378209637
Diagonals of C: [0.00809837 0.01092566 0.04225848 0.07905261 0.01976832 0.01157535
 0.06326183 0.08588073 0.03731903 0.02941576 0.02856459 0.02637713
 0.04471454 0.05412427 0.01704104 0.03735827 0.07031544]


In [105]:
#Complete F
F = S@S.T
C = np.linalg.inv(F)
print("Condition Number of F:", np.linalg.cond(F))
print("Diagonals of C:", np.diag(C))
diag = np.diag(C)
unid = [param_titles[i] for i in list(np.where(diag > 1e+3))[0]]
print("Unidentifiable:", unid)

Condition Number of F: 4020419.8306584745
Diagonals of C: [1.51996335 4.58091975 6.1646994  0.64531604 2.60929015 1.15868868
 0.74638026 0.49139222 1.95909215 4.31816193 0.60800429 1.15483001
 0.26195329 0.40740059 1.63797106 0.56827887 1.02724891 0.3323609
 4.82483401 0.43142027 0.70414324 0.23384594 0.58627892 0.50737272
 1.16113586 0.83272746 0.63936472 0.2672226  0.36216817 0.90240766
 1.01276505 0.29082176 1.01625849 0.4730218  0.46843834 1.03298106
 0.1681261  1.0218233  0.24938561 0.46498173 1.60072761 0.47110554
 0.57307417 0.2465095  0.28549264]
Unidentifiable: []
